<a href="https://www.kaggle.com/code/izzarsulynashrudin/classificationnihchestxrays?scriptVersionId=301015325" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Overview Project


Project ini menggunakan **NIH Chest X-Rays Dataset** untuk klasifikasi penyakit thoraks dan deteksi objek penyakit pada citra X-ray dada. Dataset ini berisi **112.120 gambar** sehingga sangat cocok untuk penelitian berbasis *deep learning* dalam tugas klasifikasi multi-label maupun object detection.

## Metode yang Digunakan


| Metode | Tugas                      | Metric Utama     |
| ------ | -------------------------- | ---------------- |
| CNN    | Multi-label classification | PR-AUC / ROC-AUC |
| ResNet | Multi-label classification | PR-AUC / ROC-AUC |
| YOLO   | Object detection           | mAP@0.5          |

## Link Download Dataset

[Situs Resmi](https://nihcc.app.box.com/v/ChestXray-NIHCC) 
atau 
[Kaggle Dataset](https://www.kaggle.com/datasets/nih-chest-xrays/data)

## Alasan Menggunakan Kaggle

Penggunaan Kaggle bertujuan untuk membantu proses pelatihan model karena dataset yang digunakan memiliki ukuran yang cukup besar, yaitu sebanyak **112.120 gambar**, sehingga membutuhkan akselerator GPU untuk meningkatkan performa model. Selain itu, proses pelatihan dan pengujian model juga memerlukan waktu yang cukup lama, sehingga Kaggle menjadi salah satu platform yang mendukung kebutuhan komputasi tersebut.

Oleh karena itu, saya mengucapkan terima kasih kepada pihak-pihak yang telah membantu dalam penelitian ini.

**Izzar Suly Nashrudin**


# Instalisasi Package

In [ ]:
import os
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Preprocessing

## Set Path Dataset

In [ ]:
datasetPath = "/kaggle/input/datasets/organizations/nih-chest-xrays/data"
bboxPath = "/kaggle/input/datasets/organizations/nih-chest-xrays/data/BBox_List_2017.csv"

print(os.listdir(datasetPath)[:20])

## Load dan Bersihkan Data Label

In [ ]:
labelDf = pd.read_csv(os.path.join(datasetPath, "Data_Entry_2017.csv"))

labelDf = labelDf.loc[:, ~labelDf.columns.str.contains("^Unnamed")]

labelDf = labelDf.rename(columns={
    "Image Index": "imageIndex",
    "Finding Labels": "findingLabels",
    "Follow-up #": "followUpNumber",
    "Patient ID": "patientId",
    "Patient Age": "patientAge",
    "Patient Gender": "patientGender",
    "View Position": "viewPosition",
    "OriginalImage[Width": "originalImageWidth",
    "Height]": "originalImageHeight",
    "OriginalImagePixelSpacing[x": "originalImagePixelSpacingX",
    "y]": "originalImagePixelSpacingY"
})

labelDf.head()

## Rapikan Tipe Data

In [ ]:
numericColumns = [
    "followUpNumber",
    "patientId",
    "patientAge",
    "originalImageWidth",
    "originalImageHeight",
    "originalImagePixelSpacingX",
    "originalImagePixelSpacingY"
]

for columnName in numericColumns:
    labelDf[columnName] = pd.to_numeric(labelDf[columnName], errors="coerce")

labelDf.info()

## Buat Label List dan Target Labels

In [ ]:
labelDf["labelList"] = labelDf["findingLabels"].str.split("|")

allLabels = sorted({
    labelName
    for labelList in labelDf["labelList"]
    for labelName in labelList
})

targetLabels = sorted([
    labelName
    for labelName in allLabels
    if labelName != "No Finding"
])

print(targetLabels)
print("Total target labels:", len(targetLabels))

## Multi-Label Encoding

In [ ]:
for labelName in targetLabels:
    labelDf[labelName] = labelDf["labelList"].apply(
        lambda currentLabels: int(labelName in currentLabels)
    )

labelDf["noFinding"] = labelDf["findingLabels"].apply(
    lambda x: 1 if x == "No Finding" else 0
)

labelDf.head()

In [ ]:
labelDf["labelList"]

## Kumpulkan Semua Path Gambar

In [ ]:
imageFolderPaths = []

for folderName in os.listdir(datasetPath):
    if folderName.startswith("images_"):
        folderPath = os.path.join(datasetPath, folderName, "images")
        if os.path.isdir(folderPath):
            imageFolderPaths.append(folderPath)

imageFolderPaths = sorted(imageFolderPaths)
print(imageFolderPaths)

## Buat DataFrame Path Gambar

In [ ]:
imageRows = []

for folderPath in imageFolderPaths:
    for imageName in os.listdir(folderPath):
        imageRows.append({
            "imageIndex": imageName,
            "imagePath": os.path.join(folderPath, imageName)
        })

imagePathDf = pd.DataFrame(imageRows)

print(imagePathDf.shape)
imagePathDf.head()

## Merge Label + Path

In [ ]:
masterDf = labelDf.merge(imagePathDf, on="imageIndex", how="left")
masterDf = masterDf.dropna(subset=["imagePath"]).reset_index(drop=True)

print(masterDf.shape)
print("Missing imagePath:", masterDf["imagePath"].isna().sum())
masterDf.head()

## EDA Singkat

In [ ]:
print("Total images:", len(masterDf))
print("Total patients:", masterDf["patientId"].nunique())

labelCounts = masterDf[targetLabels].sum().sort_values(ascending=False)
print(labelCounts)

Plot overview dataset

In [ ]:
labelCounts.plot(kind="bar", figsize=(12, 5))
plt.title("Distribusi Label Penyakit")
plt.show()

# Training

## Split Data Patient-Wise

In [ ]:
patientIds = masterDf["patientId"].dropna().unique()

trainPatients, tempPatients = train_test_split(
    patientIds,
    test_size=0.30,
    random_state=42
)

validPatients, testPatients = train_test_split(
    tempPatients,
    test_size=0.50,
    random_state=42
)

trainDf = masterDf[masterDf["patientId"].isin(trainPatients)].reset_index(drop=True)
validDf = masterDf[masterDf["patientId"].isin(validPatients)].reset_index(drop=True)
testDf = masterDf[masterDf["patientId"].isin(testPatients)].reset_index(drop=True)

print("Train shape:", trainDf.shape)
print("Valid shape:", validDf.shape)
print("Test shape:", testDf.shape)

## Siapkan Dataset TensorFlow

Gunakan ACCELERATOR untuk akses GPU dan CUDA

In [ ]:
imageSize = (224, 224)
batchSize = 32
autoTune = tf.data.AUTOTUNE

def decodeImage(imagePath):
    imageBytes = tf.io.read_file(imagePath)
    image = tf.image.decode_png(imageBytes, channels=3)
    image = tf.image.resize(image, imageSize)
    image = tf.cast(image, tf.float32) / 255.0
    return image

def makeDataset(dataDf, shuffle=True):
    xPaths = dataDf["imagePath"].values
    yLabels = dataDf[targetLabels].values.astype(np.float32)

    ds = tf.data.Dataset.from_tensor_slices((xPaths, yLabels))

    if shuffle:
        ds = ds.shuffle(buffer_size=min(len(dataDf), 10000), reshuffle_each_iteration=True)

    def mapFn(path, y):
        x = decodeImage(path)
        return x, y

    ds = ds.map(mapFn, num_parallel_calls=autoTune)
    ds = ds.batch(batchSize).prefetch(autoTune)
    return ds

trainDs = makeDataset(trainDf, shuffle=True)
validDs = makeDataset(validDf, shuffle=False)
testDs = makeDataset(testDf, shuffle=False)

print(trainDs)

# Model Convolutional Neural Network (CNN)

In [ ]:
numClasses = len(targetLabels)

cnnModel = keras.Sequential([
    layers.Input(shape=(imageSize[0], imageSize[1], 3)),
    layers.Conv2D(32, 3, padding="same", activation="relu"),
    layers.MaxPool2D(),
    layers.Conv2D(64, 3, padding="same", activation="relu"),
    layers.MaxPool2D(),
    layers.Conv2D(128, 3, padding="same", activation="relu"),
    layers.MaxPool2D(),
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(numClasses, activation="sigmoid")
])

cnnModel.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="binary_crossentropy",
    metrics=[
        keras.metrics.BinaryAccuracy(name="binaryAccuracy", threshold=0.5),
        keras.metrics.AUC(name="rocAuc"),
        keras.metrics.AUC(name="prAuc", curve="PR"),
        keras.metrics.Precision(name="precision", thresholds=0.5),
        keras.metrics.Recall(name="recall", thresholds=0.5),
    ]
)

cnnModel.summary()

## Training CNN

In [ ]:
cnnCallbacks = [
    keras.callbacks.ModelCheckpoint("cnnBest.keras", monitor="val_prAuc", mode="max", save_best_only=True),
    keras.callbacks.EarlyStopping(monitor="val_prAuc", mode="max", patience=3, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_prAuc", mode="max", factor=0.5, patience=2)
]

cnnHistory = cnnModel.fit(
    trainDs,
    validation_data=validDs,
    epochs=10,
    callbacks=cnnCallbacks
)

## Evaluasi CNN

In [ ]:
cnnEval = cnnModel.evaluate(testDs)
print(cnnEval)

### Metrik Evaluasi

In [ ]:

evalMetrics = cnnModel.evaluate(validDs)

print(f'Loss: {evalMetrics[0]}')
print(f'Binary Accuracy: {evalMetrics[1]}')
print(f'ROC AUC: {evalMetrics[2]}')
print(f'PR AUC: {evalMetrics[3]}')
print(f'Precision: {evalMetrics[4]}')
print(f'Recall: {evalMetrics[5]}')

### Training dan Validation Loss

In [ ]:
plt.plot(cnnHistory.history['loss'], label='Training Loss')
plt.plot(cnnHistory.history['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.title('Training dan Validation Loss')
plt.show()

# Model Residual Network (ResNet)

In [ ]:
baseModel = keras.applications.ResNet50(
    include_top=False,
    weights="imagenet",
    input_shape=(imageSize[0], imageSize[1], 3)
)

baseModel.trainable = False

inputs = keras.Input(shape=(imageSize[0], imageSize[1], 3))
x = keras.applications.resnet.preprocess_input(inputs * 255.0)
x = baseModel(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(len(targetLabels), activation="sigmoid")(x)

resnetModel = keras.Model(inputs, outputs)

resnetModel.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="binary_crossentropy",
    metrics=[
        keras.metrics.AUC(name="rocAuc"),
        keras.metrics.AUC(name="prAuc", curve="PR"),
        keras.metrics.Precision(name="precision", thresholds=0.5),
        keras.metrics.Recall(name="recall", thresholds=0.5),
    ]
)

resnetModel.summary()

## Training ResNet Tahap 1

In [ ]:
resnetCallbacks = [
    keras.callbacks.ModelCheckpoint("resnetFrozenBest.keras", monitor="val_prAuc", mode="max", save_best_only=True),
    keras.callbacks.EarlyStopping(monitor="val_prAuc", mode="max", patience=2, restore_best_weights=True)
]

resnetHistory1 = resnetModel.fit(
    trainDs,
    validation_data=validDs,
    epochs=5,
    callbacks=resnetCallbacks
)

## Fine-Tuning ResNet

In [ ]:
baseModel.trainable = True

for layer in baseModel.layers[:-30]:
    layer.trainable = False

resnetModel.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss="binary_crossentropy",
    metrics=[
        keras.metrics.AUC(name="rocAuc"),
        keras.metrics.AUC(name="prAuc", curve="PR"),
        keras.metrics.Precision(name="precision", thresholds=0.5),
        keras.metrics.Recall(name="recall", thresholds=0.5),
    ]
)

resnetHistory2 = resnetModel.fit(
    trainDs,
    validation_data=validDs,
    epochs=10,
    callbacks=[
        keras.callbacks.ModelCheckpoint("resnetFineTuneBest.keras", monitor="val_prAuc", mode="max", save_best_only=True),
        keras.callbacks.EarlyStopping(monitor="val_prAuc", mode="max", patience=3, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_prAuc", mode="max", factor=0.5, patience=2)
    ]
)

## Evaluasi ResNet

In [ ]:
resnetEval = resnetModel.evaluate(testDs)
print(resnetEval)

### Training dan Validation Loss

In [ ]:
import matplotlib.pyplot as plt

# Plot fase Frozen
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.plot(resnetHistory1.history['loss'], label='Training Loss')
plt.plot(resnetHistory1.history['val_loss'], label='Validation Loss')
plt.title('Training and Validation Loss - Fase 1 (Frozen)')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(resnetHistory1.history['rocAuc'], label='Training AUC')
plt.plot(resnetHistory1.history['val_rocAuc'], label='Validation AUC')
plt.title('Training and Validation AUC - Fase 1 (Frozen)')
plt.xlabel('Epochs')
plt.ylabel('AUC')
plt.legend()

plt.tight_layout()
plt.show()

# Plot fase Fine-Tune
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.plot(resnetHistory2.history['loss'], label='Training Loss')
plt.plot(resnetHistory2.history['val_loss'], label='Validation Loss')
plt.title('Training and Validation Loss - Fase 2 (Fine-Tune)')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(resnetHistory2.history['rocAuc'], label='Training AUC')
plt.plot(resnetHistory2.history['val_rocAuc'], label='Validation AUC')
plt.title('Training and Validation AUC - Fase 2 (Fine-Tune)')
plt.xlabel('Epochs')
plt.ylabel('AUC')
plt.legend()

plt.tight_layout()
plt.show()

# Model You Only Look Once (YOLO)

## Load Bounding Box

In [ ]:
bboxDf = pd.read_csv(bboxPath)
bboxDf = bboxDf.iloc[:, :6].copy()
bboxDf.columns = ["imageIndex", "className", "x", "y", "w", "h"]

for columnName in ["x", "y", "w", "h"]:
    bboxDf[columnName] = pd.to_numeric(bboxDf[columnName], errors="coerce")

bboxDf.head()

## Merge Bounding Box + Image Path + Patient

In [ ]:
bboxMergedDf = bboxDf.merge(imagePathDf, on="imageIndex", how="left")
bboxMergedDf = bboxMergedDf.merge(
    labelDf[["imageIndex", "patientId", "originalImageWidth", "originalImageHeight"]],
    on="imageIndex",
    how="left"
)

bboxMergedDf = bboxMergedDf.dropna(subset=["imagePath"]).reset_index(drop=True)

print(bboxMergedDf.shape)
bboxMergedDf.head()

## Split YOLO Data

In [ ]:
bboxPatientIds = bboxMergedDf["patientId"].dropna().unique()

trainPatientsYolo, tempPatientsYolo = train_test_split(
    bboxPatientIds,
    test_size=0.30,
    random_state=42
)

validPatientsYolo, testPatientsYolo = train_test_split(
    tempPatientsYolo,
    test_size=0.50,
    random_state=42
)

trainBboxDf = bboxMergedDf[bboxMergedDf["patientId"].isin(trainPatientsYolo)].reset_index(drop=True)
validBboxDf = bboxMergedDf[bboxMergedDf["patientId"].isin(validPatientsYolo)].reset_index(drop=True)
testBboxDf = bboxMergedDf[bboxMergedDf["patientId"].isin(testPatientsYolo)].reset_index(drop=True)

print(trainBboxDf.shape, validBboxDf.shape, testBboxDf.shape)

## Mapping Class YOLO

In [ ]:
classNames = sorted(bboxMergedDf["className"].dropna().unique().tolist())
classToId = {name: index for index, name in enumerate(classNames)}

print(classToId)

## Buat Folder YOLO

In [ ]:
yoloRootPath = Path("/kaggle/working/nihYOLO")

for splitName in ["train", "val", "test"]:
    (yoloRootPath / "images" / splitName).mkdir(parents=True, exist_ok=True)
    (yoloRootPath / "labels" / splitName).mkdir(parents=True, exist_ok=True)

## Konversi ke Format YOLO

In [ ]:
def writeYoloSplit(splitDf, splitName):
    groupedDf = splitDf.groupby("imageIndex")

    for imageIndex, groupDf in groupedDf:
        imagePath = groupDf["imagePath"].iloc[0]
        imageWidth = float(groupDf["originalImageWidth"].iloc[0])
        imageHeight = float(groupDf["originalImageHeight"].iloc[0])

        outputImagePath = yoloRootPath / "images" / splitName / imageIndex
        outputLabelPath = yoloRootPath / "labels" / splitName / (Path(imageIndex).stem + ".txt")

        shutil.copy(imagePath, outputImagePath)

        yoloLines = []
        for _, row in groupDf.iterrows():
            classId = classToId[row["className"]]

            x = float(row["x"])
            y = float(row["y"])
            w = float(row["w"])
            h = float(row["h"])

            xCenter = (x + w / 2.0) / imageWidth
            yCenter = (y + h / 2.0) / imageHeight
            widthNorm = w / imageWidth
            heightNorm = h / imageHeight

            yoloLines.append(
                f"{classId} {xCenter:.6f} {yCenter:.6f} {widthNorm:.6f} {heightNorm:.6f}"
            )

        outputLabelPath.write_text("\n".join(yoloLines))

writeYoloSplit(trainBboxDf, "train")
writeYoloSplit(validBboxDf, "val")
writeYoloSplit(testBboxDf, "test")

## Buat data.yaml

In [ ]:
dataYamlPath = yoloRootPath / "data.yaml"

yamlText = f"""path: {str(yoloRootPath)}
train: images/train
val: images/val
test: images/test

names:
"""

for index, className in enumerate(classNames):
    yamlText += f"  {index}: {className}\n"

dataYamlPath.write_text(yamlText)
print(yamlText)

## Install dan Train YOLO

In [ ]:
!pip -q install ultralytics

In [ ]:
from ultralytics import YOLO

yoloModel = YOLO("yolov8n.pt")

yoloModel.train(
    data=str(dataYamlPath),
    imgsz=640,
    epochs=50,
    batch=16,
    device=0
)

## Evaluasi YOLO

In [ ]:
yoloMetrics = yoloModel.val(data=str(dataYamlPath))
print(yoloMetrics)

### Plot Loss dan Metrik Pelatihan

In [ ]:
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.plot(yoloModel.history["loss"], label="Training Loss")
plt.plot(yoloModel.history["val_loss"], label="Validation Loss")
plt.title("Training vs Validation Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(yoloModel.history["map_50"], label="mAP@50")
plt.plot(yoloModel.history["val_map_50"], label="Validation mAP@50")
plt.title("mAP (mean Average Precision) vs Epochs")
plt.xlabel("Epochs")
plt.ylabel("mAP@50")
plt.legend()

plt.tight_layout()
plt.show()

### Menyimpan Model YOLO

In [ ]:
yoloModel.save("/kaggle/working/YOLOTrainedModel.pt")
print("Model YOLO telah disimpan.")